# Customer Churn Prediction — Training

**Goal:** Train and compare 5 classifiers × 2 imbalance strategies, then save the best model.

**Algorithms:** LogisticRegression, RandomForest, XGBoost, LightGBM, CatBoost
**Imbalance strategies:** `class_weight` (built-in balanced mode) and `smote` (synthetic oversampling)

> This notebook uses the project's `src/` modules under the hood but is designed to run
> fully in-process — no MLflow server required. Models and preprocessors are saved to
> `models/` for later evaluation.

## 0. Setup & Imports

In [1]:
import json
import os
import sys
import time
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings('ignore')

# ── Project root detection ────────────────────────────────────────────────
def _find_project_root():
    here = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(here, 'data', 'raw', 'cell2cell.csv')):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return os.getcwd()

ROOT = Path(_find_project_root())
sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

# ── Imports from src ───────────────────────────────────────────────────────
from src.config import (
    ALGORITHMS,
    IMBALANCE_STRATEGIES,
    RANDOM_STATE,
)
from src.data.load import load_data, split_data
from src.features.pipeline import create_preprocessing_pipeline, get_feature_names
from src.models.train import _get_class_weight_params, resolve_class

# ── Output directory for models ───────────────────────────────────────────
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)
print(f"Models will be saved to: {MODELS_DIR}")
print(f"Random state: {RANDOM_STATE}")

Project root: /Users/yusef/projects/portfolio/customer-churn-prediction
Models will be saved to: /Users/yusef/projects/portfolio/customer-churn-prediction/models
Random state: 42


## 1. Load & Split Data

Load the raw CSV, drop non-informative columns, and create a stratified 75/25 train/test split.

In [2]:
df = load_data()
print(f"Loaded data: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Target distribution:\n{df['churn'].value_counts().to_string()}")

X_train, X_test, y_train, y_test = split_data(df)
print(f"\nTrain: {len(X_train):,} samples  |  Test: {len(X_test):,} samples")

Loaded data: 71,047 rows × 69 cols
Target distribution:
churn
0    50438
1    20609

Train: 53,285 samples  |  Test: 17,762 samples


## 2. Preprocessing Pipeline

The pipeline does three things in sequence:
1. **Clean** — drops leakage column (`churndep`), median-imputes missing values, casts binaries to int
2. **Feature engineering** — ratio features (`revenue_per_mou`, `drop_call_rate`, `outcall_ratio`, `support_contact_rate`), tenure/age binning, business flags (`is_high_value`, `is_low_tenure_high_value`, `is_frequent_drops`)
3. **Scale & Encode** — `RobustScaler` on continuous columns, `OneHotEncoder` on binned columns

We fit on the **training set** and transform **both** train and test (no data leakage).

In [3]:
preprocessor = create_preprocessing_pipeline()

print("Fitting preprocessing pipeline on training data...")
X_train_processed = preprocessor.fit_transform(X_train)
feature_names = get_feature_names(preprocessor)

X_test_processed = preprocessor.transform(X_test)

print(f"Train shape after preprocessing: {X_train_processed.shape}")
print(f"Test shape after preprocessing:  {X_test_processed.shape}")
print(f"Number of features: {len(feature_names)}")
print(f"\nFeature names:\n  {'  '.join(feature_names)}")

Fitting preprocessing pipeline on training data...
Train shape after preprocessing: (53285, 84)
Test shape after preprocessing:  (17762, 84)
Number of features: 84

Feature names:
  scale__revenue  scale__mou  scale__recchrge  scale__directas  scale__overage  scale__roam  scale__changem  scale__changer  scale__mourec  scale__peakvce  scale__opeakvce  scale__outcalls  scale__incalls  scale__dropvce  scale__blckvce  scale__unansvce  scale__dropblk  scale__months  scale__eqpdays  scale__age1  scale__income  scale__setprc  ohe__tenure_group_1-2yr  ohe__tenure_group_2-4yr  ohe__tenure_group_4-6yr  ohe__tenure_group_6-12mo  ohe__age_group_35-44  ohe__age_group_45-54  ohe__age_group_55-64  ohe__age_group_65+  ohe__age_group_<25  ohe__age_group_Unknown  remainder__traintest  remainder__custcare  remainder__threeway  remainder__callfwdv  remainder__callwait  remainder__uniqsubs  remainder__actvsubs  remainder__phones  remainder__models  remainder__age2  remainder__children  remainder__credita  

## 3. Train Models

For each of the 5 algorithms, we train with **two imbalance strategies**:

| Strategy | Description |
|---|---|
| `class_weight` | Built-in balanced class weighting (each algorithm's native approach) |
| `smote` | Synthetic Minority Oversampling Technique via `imblearn` pipeline |

Results are collected into a comparison table.

In [4]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def evaluate_model(model, X_val, y_val):
    """Return dict of metrics on processed validation data."""
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]
    return {
        'roc_auc':    float(roc_auc_score(y_val, y_proba)),
        'pr_auc':     float(average_precision_score(y_val, y_proba)),
        'f1':         float(f1_score(y_val, y_pred)),
        'precision':  float(precision_score(y_val, y_pred)),
        'recall':     float(recall_score(y_val, y_pred)),
    }


results = []
best_result = None
best_pr_auc = -1.0

for algo_def in ALGORITHMS:
    algo_name = algo_def['name']
    model_class, is_sklearn = resolve_class(algo_name)
    print(f"\n{'='*60}")
    print(f"  Algorithm: {algo_name}")
    print(f"{'='*60}")

    for strategy in IMBALANCE_STRATEGIES:
        print(f"\n  --- Strategy: {strategy} ---")
        fit_start = time.time()

        # Build base params
        model_params = {'random_state': RANDOM_STATE}
        if algo_name == 'CatBoost':
            model_params = {'random_seed': RANDOM_STATE, 'verbose': 0}

        try:
            if strategy == 'class_weight':
                cw = _get_class_weight_params(algo_name, y_train)
                model_params.update(cw)
                model = model_class(**model_params)
                model.fit(X_train_processed, y_train)
            else:
                model = ImbPipeline([
                    ('smote', SMOTE(random_state=RANDOM_STATE)),
                    ('classifier', model_class(**model_params)),
                ])
                model.fit(X_train_processed, y_train)

            metrics = evaluate_model(model, X_test_processed, y_test)
            elapsed = time.time() - fit_start

            row = {
                'algorithm': algo_name,
                'strategy': strategy,
                'roc_auc': metrics['roc_auc'],
                'pr_auc': metrics['pr_auc'],
                'f1': metrics['f1'],
                'precision': metrics['precision'],
                'recall': metrics['recall'],
                'time_s': round(elapsed, 1),
                'model': model,
            }
            results.append(row)

            print(f"    ROC-AUC: {metrics['roc_auc']:.4f}  |  PR-AUC: {metrics['pr_auc']:.4f}  |  "
                  f"F1: {metrics['f1']:.4f}  |  P: {metrics['precision']:.3f}  |  "
                  f"R: {metrics['recall']:.3f}  |  {elapsed:.1f}s")

            if metrics['pr_auc'] > best_pr_auc:
                best_pr_auc = metrics['pr_auc']
                best_result = row

        except (ValueError, TypeError, RuntimeError) as e:
            print(f"    ✗ FAILED: {e}")


  Algorithm: LogisticRegression

  --- Strategy: class_weight ---
    ROC-AUC: 0.8312  |  PR-AUC: 0.5843  |  F1: 0.6626  |  P: 0.508  |  R: 0.952  |  1.3s

  --- Strategy: smote ---
    ROC-AUC: 0.8307  |  PR-AUC: 0.5822  |  F1: 0.6619  |  P: 0.511  |  R: 0.939  |  2.5s

  Algorithm: RandomForest

  --- Strategy: class_weight ---
    ROC-AUC: 0.8489  |  PR-AUC: 0.6212  |  F1: 0.6600  |  P: 0.575  |  R: 0.774  |  21.2s

  --- Strategy: smote ---
    ROC-AUC: 0.8451  |  PR-AUC: 0.6109  |  F1: 0.6214  |  P: 0.595  |  R: 0.650  |  32.9s


XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/yusef/projects/portfolio/customer-churn-prediction/.venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: '@rpath/libomp.dylib'\n  Referenced from: '/Users/yusef/projects/portfolio/customer-churn-prediction/.venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.dylib'\n  Reason: tried: '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/usr/local/lib/libomp.dylib' (no such file), '/usr/lib/libomp.dylib' (no such file)"]


## 4. Comparison & Best Model Selection

We rank models by **PR-AUC** (Precision-Recall AUC) — the most informative metric for imbalanced classification.

In [ ]:
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

# Build comparison DataFrame
comparison_df = pd.DataFrame([
    {k: r[k] for k in ['algorithm', 'strategy', 'roc_auc', 'pr_auc', 'f1', 'precision', 'recall', 'time_s']}
    for r in results
])
comparison_df = comparison_df.sort_values('pr_auc', ascending=False).reset_index(drop=True)
comparison_df['rank'] = range(1, len(comparison_df) + 1)
comparison_df = comparison_df.set_index('rank')

print("\n" + "="*90)
print("  MODEL COMPARISON  (sorted by PR-AUC ↓)")
print("="*90)
print(comparison_df.to_string(float_format=lambda x: f'{x:.4f}'))
print("="*90)

if best_result:
    print(f"\n★ Best model: {best_result['algorithm']} with '{best_result['strategy']}'")
    print(f"  PR-AUC = {best_result['pr_auc']:.4f}  |  ROC-AUC = {best_result['roc_auc']:.4f}")
    print(f"  F1 = {best_result['f1']:.4f}  |  Precision = {best_result['precision']:.3f}  |  Recall = {best_result['recall']:.3f}")

## 5. Save Best Model & Preprocessor

Persist the best model and the fitted preprocessor to disk so the evaluation notebook can load them.

In [ ]:
import joblib

if best_result:
    # Save best model
    model_path = MODELS_DIR / 'best_model.joblib'
    joblib.dump(best_result['model'], model_path)
    print(f"✓ Best model saved to: {model_path}")

    # Save preprocessor
    preproc_path = MODELS_DIR / 'preprocessor.joblib'
    joblib.dump(preprocessor, preproc_path)
    print(f"✓ Preprocessor saved to: {preproc_path}")

    # Save feature names
    feat_path = MODELS_DIR / 'feature_names.txt'
    with open(feat_path, 'w') as f:
        f.write('\n'.join(feature_names))
    print(f"✓ Feature names saved to: {feat_path}")

    # Save comparison table
    comp_path = MODELS_DIR / 'comparison_results.csv'
    comparison_df.to_csv(comp_path)
    print(f"✓ Comparison table saved to: {comp_path}")

    # Save best model metadata
    meta = {
        'algorithm': best_result['algorithm'],
        'strategy': best_result['strategy'],
        'pr_auc': best_result['pr_auc'],
        'roc_auc': best_result['roc_auc'],
        'f1': best_result['f1'],
        'precision': best_result['precision'],
        'recall': best_result['recall'],
        'n_features': len(feature_names),
        'n_train': len(X_train),
        'n_test': len(X_test),
    }
    meta_path = MODELS_DIR / 'best_model_metadata.json'
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)
    print(f"✓ Metadata saved to: {meta_path}")

    print(f"\n{'='*60}")
    print("  Training complete. Ready for evaluation notebook.")
    print(f"{'='*60}")
else:
    print("✗ No model was successfully trained.")

## 6. Summary

| Metric | Value |
|---|---|
| Algorithms trained | 5 (LogisticRegression, RandomForest, XGBoost, LightGBM, CatBoost) |
| Strategies per algorithm | 2 (class_weight, smote) |
| Best model | See output above |
| Output artifacts | `models/best_model.joblib`, `models/preprocessor.joblib`, `models/comparison_results.csv` |

Next: Open `notebooks/03-eval.ipynb` to load the best model and produce evaluation plots + SHAP explanations.